<a href="https://colab.research.google.com/github/Asuskf/from-nlp-to-agents/blob/tokens/tokens/Chain%20Rule/LLM_Probability_Chain_Rule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The Mathematical Foundation: **The Chain Rule of Probability in LLMs**

A Large Language Model (LLM) has a single core mathematical objective: to estimate the joint probability distribution of a sequence of tokens.

If we have a sequence of tokens $W = (w_1, w_2, \dots, w_T)$, the probability of that sequence existing is defined by the **chain rule of probability**:

$$P(W) = P(w_1, w_2, \dots, w_T) = \prod_{t=1}^T P(w_t | w_1, w_2, \dots, w_{t-1})$$

The model does not predict the text $W$ all at once. Instead, it is trained to approximate the conditional probability $P(w_t | w_{<t}; \theta)$, where $\theta$ represents the neural network weights (parameters). At time step $t$, the model takes all previous context $(w_1 \dots w_{t-1})$ and outputs a probability distribution over its entire vocabulary for the next token $w_t$.

In [2]:
from pydantic import BaseModel, Field
from typing import List, Dict

class TokenDistribution(BaseModel):
    """Represents a probability distribution over a vocabulary for the next token."""
    probabilities: Dict[str, float] = Field(
        ...,
        description="Dictionary mapping a token to its predicted probability."
    )

class LLMSimulator(BaseModel):
    """Simulates an LLM predicting conditional probabilities."""

    # This represents our mocked neural network weights (theta)
    knowledge_base: Dict[str, TokenDistribution] = Field(
        ...,
        description="Pre-defined distributions based on string contexts."
    )

    def get_distribution(self, context: List[str]) -> TokenDistribution:
        """Returns the probability distribution given the current context (w_<t)."""
        context_str = " ".join(context)
        # Return the distribution if known, otherwise return an unknown token distribution
        return self.knowledge_base.get(
            context_str,
            TokenDistribution(probabilities={"<unk>": 1.0})
        )


if __name__ == "__main__":
    # 1. Define our mock "weights" (theta) for specific Spanish contexts
    mock_weights = {
        "": TokenDistribution(probabilities={"El": 0.5, "La": 0.3, "Un": 0.2}),
        "El": TokenDistribution(probabilities={"gato": 0.6, "perro": 0.3, "coche": 0.1}),
        "El gato": TokenDistribution(probabilities={"duerme": 0.8, "come": 0.1, "salta": 0.1})
    }

    # 2. Instantiate our model simulator
    model = LLMSimulator(knowledge_base=mock_weights)

    # 3. Define the target sequence W
    sequence_W = ["El", "gato", "duerme"]

    # 4. Initialize variables
    joint_probability_P_W = 1.0
    current_context = []

    print(f"Evaluating Sequence W = {sequence_W}\n")
    print("-" * 50)

    # 5. Iterate through each time step (The Product operator Π)
    for t, target_token in enumerate(sequence_W, start=1):

        # Forward pass: get vocabulary distribution conditioned on history
        distribution = model.get_distribution(current_context)

        # Extract probability for the specific target token
        prob_token_t = distribution.probabilities.get(target_token, 0.0)

        print(f"Step t={t}:")
        print(f"  Context (w_<{t}): {current_context}")
        print(f"  Generated Distribution: {distribution.probabilities}")
        print(f"  P('{target_token}' | context) = {prob_token_t}")
        print("-" * 50)

        # Multiply current probability (Chain Rule calculation)
        joint_probability_P_W *= prob_token_t

        # Append current token to context for the next time step
        current_context.append(target_token)

    print("\nFinal Result:")
    print(f"P(W) = {joint_probability_P_W:.3f} (or {joint_probability_P_W * 100:.1f}%)")

Evaluating Sequence W = ['El', 'gato', 'duerme']

--------------------------------------------------
Step t=1:
  Context (w_<1): []
  Generated Distribution: {'El': 0.5, 'La': 0.3, 'Un': 0.2}
  P('El' | context) = 0.5
--------------------------------------------------
Step t=2:
  Context (w_<2): ['El']
  Generated Distribution: {'gato': 0.6, 'perro': 0.3, 'coche': 0.1}
  P('gato' | context) = 0.6
--------------------------------------------------
Step t=3:
  Context (w_<3): ['El', 'gato']
  Generated Distribution: {'duerme': 0.8, 'come': 0.1, 'salta': 0.1}
  P('duerme' | context) = 0.8
--------------------------------------------------

Final Result:
P(W) = 0.240 (or 24.0%)



The final output **P(W) = 0.240 (or 24.0%)** means our model has a 24% chance of generating the exact sequence *"El gato duerme"* from scratch.

This joint probability is calculated by multiplying the conditional probabilities at each step (the **Chain Rule**):

1. **`P('El' | empty context) = 0.5`** (50% chance to start with "El")
2. **`P('gato' | 'El') = 0.6`** (60% chance to predict "gato" given "El")
3. **`P('duerme' | 'El gato') = 0.8`** (80% chance to predict "duerme" given "El gato")

**Calculation:** $0.5 \times 0.6 \times 0.8 = 0.24$

**Key Takeaway:**
As sequences get longer, the joint probability strictly decreases (multiplying fractions yields smaller numbers). To prevent these numbers from becoming so small that computers round them to zero (*underflow*), real-world LLMs use **log-probabilities (log-probs)**, adding values instead of multiplying them.